In [2]:
import re
import gspread
import pandas as pd

from google.oauth2.service_account import Credentials

In [34]:
SERVICE_ACCOUNT_FILE = 'key/Credentials.json'
SCOPES = ['https://www.googleapis.com/auth/spreadsheets',
          'https://www.googleapis.com/auth/drive']
creds = Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=SCOPES
)
client = gspread.authorize(creds)

sheet = client.open('NEW Akulaku - AI Kula Scene Classification (2025)').worksheet('NEW 12.14')
data = sheet.get_all_records()

df = pd.DataFrame(data[0:], columns=data[0])

In [35]:
# cleaning the columns
df_clean = df.copy()

def remove_mandarin(text):
    return re.sub('r[\u4e00-\u9fff]+', '', str(text))

df_clean.columns = (
    df_clean.columns
    .map(remove_mandarin)
    .str.replace('\n', '')
    .str.replace(' ','_')
    .str.replace(r'[^A-Za-z0-9_()-]', '', regex=True)
    .str.replace('-','_')
    .str.lower()
    .str.strip('_')
)

df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 923 entries, 0 to 922
Data columns (total 57 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0                                923 non-null    object
 1   no                           923 non-null    int64 
 2   create_time                  923 non-null    object
 3   classification_business      923 non-null    object
 4   faq_status                   923 non-null    object
 5   old_code                     923 non-null    object
 6   new_code                     923 non-null    object
 7   kategori_1_(cn)              923 non-null    object
 8   kategori_1_(en)              923 non-null    object
 9   kategori_1_(id)              923 non-null    object
 10  kategori_2_(cn)              923 non-null    object
 11  kategori_2_(en)              923 non-null    object
 12  kategori_2_(id)              923 non-null    object
 13  kategori_3_(cn)              923 no

In [36]:
# The raw data
raw = [
    '''
1-18-62-422
1134-1156-1179-807
1334-1069
1334-1335-1049
1334-1335-1051
1334-1335-1059
1334-1335-1062
1334-1335-1065
1334-1335-1072
1334-1335-1107
1334-1335-1116
1334-1335-1121
1334-1335-1123
1334-1335-1134
1344-1349-1411-233
1344-1356-1418-260
1344-1365-1427-278
1344-1398-1460-653
1344-1400-1462-677
1344-1402-1464-680
1662-1663-1664-996
289-1527-1528-922
289-1562-1563-943
289-1667-1668-998
289-1714-1715-1160
289-1770-1771-1188
289-1821-1822-1217
289-292-446-173
289-303-460-349
289-308-465-411
289-309-466-412
289-317-474-477
289-326-483-503
289-327-484-505
289-328-485-506
289-332-489-510
289-338-495-516
289-364-521-570
289-369-526-713
289-369-526-777
289-374-531-610
289-380-537-622
289-382-539-624
289-397-554-652
289-398-555-656
289-409-566-712
289-410-567-715
289-414-571-735
289-415-572-736
289-416-573-737
289-419-576-751
289-423-580-772
289-424-581-778
289-425-582-816
289-433-590-834
289-435-592-846
289-437-594-849
695-701-708-740
710-711-717-440
710-713-719-720
710-714-720-727
710-715-721-748
738-1608-1609-967
738-739-758-121
738-755-774-837
74-169-276-815
794-797-811-214
794-805-819-752
794-807-821-810
872-1721-1163
872-1736-1737-1171
872-1787-1788-1200
872-890-985-243
872-900-995-372
872-915-1010-494
872-922-1017-527
872-949-1044-769
872-951-1046-774
'''
]

code_list = pd.DataFrame(raw[0].split('\n'), columns=['code_list'])

code_list = code_list.replace('','-').drop_duplicates(keep=False)
code_list

,code_list
1,1-18-62-422
2,1134-1156-1179-807
3,1334-1069
4,1334-1335-1049
5,1334-1335-1051
...,...
74,872-900-995-372
75,872-915-1010-494
76,872-922-1017-527
77,872-949-1044-769


In [37]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 923 entries, 0 to 922
Data columns (total 57 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0                                923 non-null    object
 1   no                           923 non-null    int64 
 2   create_time                  923 non-null    object
 3   classification_business      923 non-null    object
 4   faq_status                   923 non-null    object
 5   old_code                     923 non-null    object
 6   new_code                     923 non-null    object
 7   kategori_1_(cn)              923 non-null    object
 8   kategori_1_(en)              923 non-null    object
 9   kategori_1_(id)              923 non-null    object
 10  kategori_2_(cn)              923 non-null    object
 11  kategori_2_(en)              923 non-null    object
 12  kategori_2_(id)              923 non-null    object
 13  kategori_3_(cn)              923 no

In [39]:
# Imputating the holy faq
kategori = df_clean.iloc[:,[6,9,12,15,18,21,47]]

# Comparing FAQ
result = code_list.merge(kategori, left_on='code_list', right_on='new_code', how='left')
result.to_csv('temp/faq.csv')

In [51]:
# Pull
# sheet = client.open('NEW Akulaku - AI Kula Scene Classification (2025)').worksheet('Chit-Chat Added')
# data = sheet.get_all_records()

# df_chit_chat = pd.DataFrame(data[0:], columns=data[0])
# df_chit_chat.to_csv('temp/faq_chit_chat.csv')
df_chit_chat = pd.read_csv('temp/faq_chit_chat.csv')

# clean the column
df_chit_chat.columns = (
    df_chit_chat.columns
    .map(remove_mandarin)
    .str.replace('\n', '')
    .str.replace(' ','_')
    .str.replace(r'[^A-Za-z0-9_()-]', '', regex=True)
    .str.replace('-','_')
    .str.lower()
    .str.strip('_')
)

# Imputation
df_chit_chat = df_chit_chat.iloc[:,[3,5,6,7,10,13,14]]

# removing extra spaces
df_chit_chat = df_chit_chat.apply(lambda x: x.str.strip() if x.dtype == 'object' else x)

df_chit_chat

AttributeError: Can only use .str accessor with string values!

In [32]:
impute_chit_chat = result[result['new_code'].isna()]
impute_chit_chat

,code_list,new_code,kategori_1_(id),kategori_2_(id),kategori_3_(id),idbackground_detail__id,idexample
2,1334-1069,NaN,NaN,NaN,NaN,NaN,NaN
3,1334-1335-1049,NaN,NaN,NaN,NaN,NaN,NaN
4,1334-1335-1051,NaN,NaN,NaN,NaN,NaN,NaN
5,1334-1335-1059,NaN,NaN,NaN,NaN,NaN,NaN
6,1334-1335-1062,NaN,NaN,NaN,NaN,NaN,NaN
7,1334-1335-1065,NaN,NaN,NaN,NaN,NaN,NaN
8,1334-1335-1072,NaN,NaN,NaN,NaN,NaN,NaN
9,1334-1335-1107,NaN,NaN,NaN,NaN,NaN,NaN
10,1334-1335-1116,NaN,NaN,NaN,NaN,NaN,NaN
11,1334-1335-1121,NaN,NaN,NaN,NaN,NaN,NaN


In [48]:
# Comparing chit-chat
result_chit_chat = impute_chit_chat.merge(df_chit_chat, left_on='code_list', right_on='code', how='left')
result_chit_chat

,code_list,new_code,kategori_1_(id),kategori_2_(id),kategori_3_(id),idbackground_detail__id,idexample,code,penjelasan_skenario_2,penjelasan_skenario_3,id_example_(id),answer_(id),testing_online_chit_chat111024,answer_false
0,1334-1069,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1334-1335-1049,NaN,NaN,NaN,NaN,NaN,NaN,1334-1335-1049,Chitchat,Basa basi,Min,"Iya Kak, ada yang bisa Kula bantu?",False,"Maaf, kula tidak mengerti apa yang Anda maksud..."
2,1334-1335-1051,NaN,NaN,NaN,NaN,NaN,NaN,1334-1335-1051,Chitchat,Basa basi,Mau tanya,Hai Kak. Terima kasih sudah chat Kula. Ada yan...,True,NaN
3,1334-1335-1059,NaN,NaN,NaN,NaN,NaN,NaN,1334-1335-1059,Chitchat,Basa basi,Sudah,Baik Kak. Terima kasih\n\nBaik Kak. Ada yang b...,False,https://prnt.sc/4TEs4LDMKrJV \nHai Kak! Mohon ...
4,1334-1335-1062,NaN,NaN,NaN,NaN,NaN,NaN,1334-1335-1062,Chitchat,Basa basi,tapi kok sdah ada tagihan,Mohon maaf ya Kak sudah buat Kakak tidak nyama...,NaN,NaN
5,1334-1335-1065,NaN,NaN,NaN,NaN,NaN,NaN,1334-1335-1065,Chitchat,Basa basi,siang,Halo Kak. Ada yang bisa Kula bantu?,True,NaN
6,1334-1335-1072,NaN,NaN,NaN,NaN,NaN,NaN,1334-1335-1072,Chitchat,Basa basi,hallo,Halo Kak. Terima kasih sudah chat Kula,True,NaN
7,1334-1335-1107,NaN,NaN,NaN,NaN,NaN,NaN,1334-1335-1107,Chitchat,Basa basi,selamat malam,"Makasih sudah chat Kula, apa masih ada yang bi...",True,NaN
8,1334-1335-1116,NaN,NaN,NaN,NaN,NaN,NaN,1334-1335-1116,Chitchat,Basa basi,pagi,Halo Kak. Ada yang bisa Kula bantu?,True,NaN
9,1334-1335-1121,NaN,NaN,NaN,NaN,NaN,NaN,1334-1335-1121,Chitchat,Basa basi,minta tolong,"Iya Kak, ada yang bisa Kula bantu?",True,NaN
